# How AI Remembers: Understanding LSTM Networks

When you read a sentence, you remember what came before. AI needs this ability too!

**LSTM** (Long Short-Term Memory) networks give AI the ability to remember important information and forget what's not needed.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

import plotly.io as pio
pio.renderers.default = "notebook"

print("✓ Ready to explore how AI remembers!")

## The Problem: AI Without Memory

Imagine reading: **"The clouds are dark. It will probably ___"**

You know the answer is **"rain"** because you remember the dark clouds.

Basic AI looks at each word separately and forgets what came before. It might guess "sing" or "dance" because it doesn't remember the context!

**LSTM networks solve this by giving AI a memory system with three simple controls:**

| Control | What it does | Example |
|---------|--------------|----------|
| **Forget Gate** | Decides what to forget | "We changed topics, forget the old subject" |
| **Input Gate** | Decides what new info to remember | "This word is important, save it" |
| **Output Gate** | Decides what to use right now | "For this prediction, focus on the weather" |

In [ ]:
# =============================================================================
# INTERACTIVE LSTM VISUALIZATION
# =============================================================================

class SimpleLSTMVisualization:
    """Simple, clean LSTM visualization"""
    
    def __init__(self):
        self.steps = [
            {
                'name': 'Overview: The Memory Cell',
                'active': ['forget_gate', 'input_gate', 'memory', 'output_gate'],
                'explanation': 'An LSTM is like a smart filing cabinet. It has three gates that control what information flows in, what stays, and what comes out.',
                'analogy': 'Think of it like your desk: you decide what papers to throw away (forget), what new papers to file (input), and what to pull out for your current task (output).'
            },
            {
                'name': 'Step 1: The Forget Gate',
                'active': ['forget_gate', 'memory'],
                'explanation': 'First, the AI looks at old memories and decides what is no longer relevant. The forget gate scores each memory from 0 (forget it) to 1 (keep it).',
                'analogy': 'Like cleaning your desk: "Do I still need these old notes? The meeting is over, so I can toss them."'
            },
            {
                'name': 'Step 2: The Input Gate',
                'active': ['input_gate', 'memory'],
                'explanation': 'Next, the AI looks at new information and decides what is worth remembering. Important things get stored, unimportant things are ignored.',
                'analogy': 'Like receiving new papers: "This report is important for my project, I\'ll file it. This junk mail goes straight to recycling."'
            },
            {
                'name': 'Step 3: Update Memory',
                'active': ['memory'],
                'explanation': 'Now the memory is updated: old irrelevant stuff is removed, new important stuff is added. This is the AI\'s working memory.',
                'analogy': 'Your filing cabinet now has the right mix of old and new information, ready to help you with your work.'
            },
            {
                'name': 'Step 4: The Output Gate',
                'active': ['memory', 'output_gate'],
                'explanation': 'Finally, the AI decides which memories are relevant for the current task. Not everything in memory is needed right now.',
                'analogy': 'You have lots of files, but for this email you only need the budget numbers. The output gate pulls just what you need.'
            }
        ]
        
        self.create_interface()
    
    def create_interface(self):
        # Simple step buttons
        self.step_slider = widgets.IntSlider(
            value=0, min=0, max=4, step=1,
            description='Step:',
            style={'description_width': '50px'},
            layout=widgets.Layout(width='400px'),
            continuous_update=False
        )
        
        self.prev_btn = widgets.Button(description='← Back', layout=widgets.Layout(width='80px'))
        self.next_btn = widgets.Button(description='Next →', layout=widgets.Layout(width='80px'))
        
        self.info_display = widgets.HTML()
        self.plot_output = widgets.Output()
        
        self.step_slider.observe(self._on_change, 'value')
        self.prev_btn.on_click(lambda b: self._change_step(-1))
        self.next_btn.on_click(lambda b: self._change_step(1))
        
        controls = widgets.HBox([
            self.prev_btn, self.step_slider, self.next_btn
        ], layout=widgets.Layout(justify_content='center', gap='10px', margin='10px 0'))
        
        display(controls)
        display(self.info_display)
        display(self.plot_output)
        
        self._update()
    
    def _on_change(self, change):
        self._update()
    
    def _change_step(self, delta):
        new_val = self.step_slider.value + delta
        if 0 <= new_val <= 4:
            self.step_slider.value = new_val
    
    def _update(self):
        step_idx = self.step_slider.value
        step = self.steps[step_idx]
        
        self.prev_btn.disabled = (step_idx == 0)
        self.next_btn.disabled = (step_idx == 4)
        
        # Simple info box
        self.info_display.value = f"""
        <div style="background: #f0f9ff; padding: 20px; border-radius: 10px; margin: 10px 0; border: 1px solid #bae6fd;">
            <h3 style="margin: 0 0 10px 0; color: #0369a1;">{step['name']}</h3>
            <p style="margin: 0 0 15px 0; font-size: 15px; color: #334155;">{step['explanation']}</p>
            <div style="background: #fef3c7; padding: 12px; border-radius: 6px; border-left: 4px solid #f59e0b;">
                <strong style="color: #92400e;">💡 Real-world analogy:</strong><br>
                <span style="color: #78350f;">{step['analogy']}</span>
            </div>
        </div>
        """
        
        with self.plot_output:
            clear_output(wait=True)
            self._draw_lstm(step['active'])
    
    def _draw_lstm(self, active_components):
        """Draw simple, clear LSTM diagram"""
        
        fig = go.Figure()
        
        # Component positions (spread out, no overlap)
        components = {
            'forget_gate': {'x': 1, 'y': 2, 'label': 'Forget\nGate', 'color': '#ef4444', 'icon': '🗑️'},
            'input_gate': {'x': 3, 'y': 2, 'label': 'Input\nGate', 'color': '#3b82f6', 'icon': '📥'},
            'memory': {'x': 5, 'y': 2, 'label': 'Memory', 'color': '#f59e0b', 'icon': '🧠'},
            'output_gate': {'x': 7, 'y': 2, 'label': 'Output\nGate', 'color': '#8b5cf6', 'icon': '📤'}
        }
        
        # Draw each component
        for name, comp in components.items():
            is_active = name in active_components
            
            # Box size
            size = 55 if is_active else 40
            opacity = 1.0 if is_active else 0.3
            color = comp['color'] if is_active else '#d1d5db'
            
            # Draw box
            fig.add_trace(go.Scatter(
                x=[comp['x']], y=[comp['y']],
                mode='markers',
                marker=dict(
                    size=size,
                    color=color,
                    symbol='square',
                    line=dict(color='white', width=3),
                    opacity=opacity
                ),
                showlegend=False,
                hoverinfo='skip'
            ))
            
            # Icon above
            if is_active:
                fig.add_annotation(
                    x=comp['x'], y=comp['y'] + 0.9,
                    text=comp['icon'],
                    showarrow=False,
                    font=dict(size=28)
                )
            
            # Label below
            label_color = comp['color'] if is_active else '#9ca3af'
            fig.add_annotation(
                x=comp['x'], y=comp['y'] - 0.9,
                text=f"<b>{comp['label']}</b>",
                showarrow=False,
                font=dict(size=12 if is_active else 10, color=label_color)
            )
        
        # Draw arrows between active components
        arrow_pairs = [
            ('forget_gate', 'memory'),
            ('input_gate', 'memory'),
            ('memory', 'output_gate')
        ]
        
        for start_name, end_name in arrow_pairs:
            if start_name in active_components and end_name in active_components:
                start = components[start_name]
                end = components[end_name]
                
                fig.add_annotation(
                    x=end['x'] - 0.5, y=end['y'],
                    ax=start['x'] + 0.5, ay=start['y'],
                    xref='x', yref='y',
                    axref='x', ayref='y',
                    showarrow=True,
                    arrowhead=2,
                    arrowsize=1.5,
                    arrowwidth=3,
                    arrowcolor='#64748b'
                )
        
        # Input arrow
        if 'forget_gate' in active_components or 'input_gate' in active_components:
            fig.add_annotation(
                x=0.3, y=2,
                text='<b>New<br>Input</b>',
                showarrow=True,
                ax=-0.5, ay=2,
                axref='x', ayref='y',
                arrowhead=2,
                arrowsize=1.5,
                arrowwidth=3,
                arrowcolor='#22c55e',
                font=dict(size=11, color='#22c55e')
            )
        
        # Output arrow
        if 'output_gate' in active_components:
            fig.add_annotation(
                x=8.2, y=2,
                ax=7.5, ay=2,
                axref='x', ayref='y',
                text='<b>Output</b>',
                showarrow=True,
                arrowhead=2,
                arrowsize=1.5,
                arrowwidth=3,
                arrowcolor='#22c55e',
                font=dict(size=11, color='#22c55e')
            )
        
        # Layout
        fig.update_layout(
            height=350,
            width=800,
            xaxis=dict(visible=False, range=[-1, 9]),
            yaxis=dict(visible=False, range=[0.5, 3.5]),
            margin=dict(l=20, r=20, t=20, b=20),
            paper_bgcolor='white',
            plot_bgcolor='white'
        )
        
        fig.show()

# Run the visualization
print("Use the slider or buttons to step through how an LSTM works:")
print()
lstm_viz = SimpleLSTMVisualization()

## Summary: Why LSTM Matters

LSTM networks are used everywhere AI needs to understand sequences:

- **Language**: Understanding sentences, translating text, chatbots
- **Speech**: Converting voice to text, voice assistants
- **Time series**: Stock prices, weather prediction, sensor data
- **Music**: Generating melodies, understanding rhythm patterns

The key insight: **AI needs to remember context to make good predictions**, just like humans do.

The three gates work together:
1. **Forget Gate**: Clear out old, irrelevant information
2. **Input Gate**: Store new, important information  
3. **Output Gate**: Use the right memories for the current task